# XAI-ChestDiag - build 384px image cache (CPU only, no GPU quota)

Input: NIH Chest X-rays. Output: images_384.zip


In [ ]:
RESIZE = 384

In [ ]:
import os
os.makedirs('/kaggle/working/XAI-ChestDiag/src', exist_ok=True)
os.chdir('/kaggle/working/XAI-ChestDiag')

In [ ]:
%%writefile src/resize_cache.py
"""Optional speed-up: pre-resize all NIH images once (1024x1024 -> 256x256).

Decoding 112k full-resolution PNGs every epoch is usually the bottleneck.
After running this, set `data.image_root` in the config to the output folder.
Bounding-box coordinates stay valid because the pointing game rescales using
the original width/height stored in Data_Entry_2017.csv.

Usage:
    python -m src.resize_cache --src data/nih/images --dst data/nih/images_256 --size 256
"""
from __future__ import annotations

import argparse
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import cv2


def _one(args):
    src, dst, size = args
    if Path(dst).exists():
        return
    img = cv2.imread(src, cv2.IMREAD_GRAYSCALE)
    cv2.imwrite(dst, cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True)
    ap.add_argument("--dst", required=True)
    ap.add_argument("--size", type=int, default=256)
    ap.add_argument("--workers", type=int, default=8)
    a = ap.parse_args()
    dst = Path(a.dst)
    dst.mkdir(parents=True, exist_ok=True)
    jobs = [(str(p), str(dst / p.name), a.size) for p in Path(a.src).rglob("*.png")]
    print(f"{len(jobs)} images -> {dst}")
    with ProcessPoolExecutor(a.workers) as ex:
        for i, _ in enumerate(ex.map(_one, jobs, chunksize=256)):
            if i % 10000 == 0:
                print(i)


if __name__ == "__main__":
    main()


In [ ]:
open('src/__init__.py','w').close()
import glob
hits = glob.glob('/kaggle/input/**/Data_Entry_2017.csv', recursive=True)
NIH = os.path.dirname(sorted(hits, key=len)[0]); IMG = f'/tmp/images_{RESIZE}'
print(NIH)

In [ ]:
!python -m src.resize_cache --src {NIH} --dst {IMG} --size {RESIZE} --workers {os.cpu_count()}

In [ ]:
import zipfile
n = 0
with zipfile.ZipFile(f'/kaggle/working/images_{RESIZE}.zip', 'w', zipfile.ZIP_STORED) as z:
    for f in sorted(os.listdir(IMG)):
        z.write(os.path.join(IMG, f), f'images_{RESIZE}/{f}'); n += 1
print(n, 'images zipped')
!ls -la /kaggle/working